# DIMER Table Intelligence Workshop
## Table Detection → Structure Recognition → Table Reconstruction → TAPAS QA

**Notebook profile:** `TASK-INFERENCE`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Workflow scope:** `COMPOSED-PIPELINE`  
**Standalone:** yes  
**Canonical workflow:** frozen inference only

This workshop treats Table Intelligence as a system:

```text
document page
→ table detection
→ table crop
→ row/column structure
→ reconstructed cell grid
→ cell text assignment
→ structured table
→ TAPAS question answering
```

Models:

1. **Table Transformer Detection**
2. **Table Transformer Structure Recognition v1.1-all**
3. **TAPAS Large WTQ**

The canonical corpus is the public-domain **SciTSR-PD** scientific-table dataset. Its logical cell text is used as a controlled text provider so this notebook can isolate table geometry and QA without mixing in OCR quality.

### Learning objectives

You will:

- distinguish detection, structure recognition and table QA;
- reconstruct a rectangular table grid from predicted rows/columns;
- trace coordinate transforms from page → crop → table;
- assign source cell text into a predicted grid;
- ask lookup and aggregation questions with TAPAS;
- compare gold-table, structure-only and full end-to-end QA;
- build a stage-by-stage error waterfall; and
- understand where OCR enters a production table-extraction system.

> **Important boundary:** Table Transformer models recover geometry, not text. The canonical workflow uses SciTSR annotation text, not OCR inference. OCR / Document Extraction is the next notebook in the Document Intelligence track.

## 1. Three different tasks

### Table detection
**Where is the table on the page?**

### Structure recognition
**Where are rows, columns and spanning cells inside the table?**

### Table QA
**What does the reconstructed table say?**

A correct final answer depends on all upstream interfaces.

## 2. Error waterfall

The same TAPAS questions are evaluated through three paths:

```text
Path 1:
gold logical table → TAPAS

Path 2:
gold table crop → predicted structure → reconstructed table → TAPAS

Path 3:
page → predicted table crop → predicted structure → reconstructed table → TAPAS
```

This separates QA-model error from structure error and table-detection error.

## 3. Configuration

The defaults define the canonical `Run all` path.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}

DETECTION_THRESHOLD = 0.90  # @param {type:"number"}
STRUCTURE_THRESHOLD = 0.50  # @param {type:"number"}
CROP_PADDING = 10  # @param {type:"integer"}
GRID_NMS_IOU = 0.50  # @param {type:"number"}

END_TO_END_TABLES = 10  # @param {type:"integer"}
QUESTIONS_PER_TABLE = 5

RUN_COMPLEX_STRUCTURE_PROBE = True  # @param {type:"boolean"}
COMPLEX_STRUCTURE_TABLES = 3  # @param {type:"integer"}

OUTPUT_DIR = "outputs/table_intelligence"

if not 0 <= DETECTION_THRESHOLD <= 1:
    raise ValueError("DETECTION_THRESHOLD must be in [0,1]")
if not 0 <= STRUCTURE_THRESHOLD <= 1:
    raise ValueError("STRUCTURE_THRESHOLD must be in [0,1]")
if END_TO_END_TABLES != 10:
    raise ValueError("Canonical workflow uses exactly 10 tables")
print({
    "detection_threshold":DETECTION_THRESHOLD,
    "structure_threshold":STRUCTURE_THRESHOLD,
    "crop_padding":CROP_PADDING,
    "tables":END_TO_END_TABLES,
})

## 4. Runtime

The three live DIMER carriers share the same Python/PyTorch runtime family. `datasets` and `pyarrow` are used only to read the small pinned SciTSR-PD parquet files.

A Tesla T4 is recommended because TAPAS Large is approximately 1.35 GB.

In [ ]:
import importlib.metadata as importlib_metadata
import subprocess, sys

PINS = {
    "torch":"2.14.0",
    "torchvision":"0.29.0",
    "torchaudio":"2.11.0",
    "transformers":"4.57.6",
    "safetensors":"0.8.0",
    "numpy":"2.5.3",
    "pillow":"11.3.0",
    "huggingface-hub":"0.36.2",
    "datasets":"4.1.1",
    "pyarrow":"25.0.1",
    "pandas":"2.2.3",
}

def version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

before={k:version(k) for k in PINS}
needed=[f"{k}=={v}" for k,v in PINS.items() if before[k]!=v]
if needed:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*needed])

after={k:version(k) for k in PINS}
bad={k:(after[k],v) for k,v in PINS.items() if after[k]!=v}
if bad:
    raise RuntimeError(f"Pinned install failed: {bad}")

import numpy as np
import pandas as pd
import torch
import torchvision
import transformers
import datasets
import pyarrow as pa
from PIL import Image, ImageDraw, ImageFont, ImageOps

DEVICE="cuda:0" if torch.cuda.is_available() else "cpu"
RUNTIME={
    "python":sys.version.split()[0],
    "torch":torch.__version__,
    "torchvision":torchvision.__version__,
    "transformers":transformers.__version__,
    "datasets":datasets.__version__,
    "numpy":np.__version__,
    "pandas":pd.__version__,
    "pillow":importlib_metadata.version("pillow"),
    "device":DEVICE,
}
if torch.cuda.is_available():
    RUNTIME["gpu_name"]=torch.cuda.get_device_name(0)
print(RUNTIME)

## 5. Immutable model provenance

Every model is staged only from its exact immutable revision and verified against the DIMER manifest before loading.

No upstream `.bin` file is used.

In [ ]:
import hashlib, json, time, gc, math, random
from pathlib import Path
from huggingface_hub import hf_hub_download

DETECTION_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "table-transformer-detection", "modelId": "microsoft/table-transformer-detection", "revision": "2357cbe2b5a5d1c03e54f32764f06058933b65ab", "files": [{"path": "README.md", "bytes": 1174, "sha256": "c91e7f8199313c4d24b09e73a2f6df3141268666969346cb7b94ccb59900f5dd"}, {"path": "config.json", "bytes": 1228, "sha256": "ed5b93df2c3a59d473ddea853553a6d545d52bd4e9f8f72bf40b8a974aba4c1d"}, {"path": "model.safetensors", "bytes": 115317516, "sha256": "8f1aa73170102c038d40155e2734b343bf07e0fe12594228a8590943b01dccf7"}, {"path": "preprocessor_config.json", "bytes": 273, "sha256": "86a8837ae440456b0a9aef788b064921df29c20f0b67040954ce5c2fbd352c4f"}], "totalBytes": 115320191}""")
STRUCTURE_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "table-transformer-structure-v1.1-all", "modelId": "microsoft/table-transformer-structure-recognition-v1.1-all", "revision": "7587a7ef111d9dcbf8ac695f1376ab7014340a0c", "files": [{"path": "README.md", "bytes": 1056, "sha256": "01eed360e0f54ad297ab347cfef208d68d8ac2c108b808147bb7af9f8d9132c8"}, {"path": "config.json", "bytes": 76761, "sha256": "17a8a6edfb9e394263fa6ba9b82176ebccdfcc5d6cd29121ec91572c7d6be22c"}, {"path": "model.safetensors", "bytes": 115437156, "sha256": "9df416575a3a36ebd0129342d4f597f14d6e5170268f3d52d28584ab4466a501"}, {"path": "preprocessor_config.json", "bytes": 374, "sha256": "eead409bb80e36ae85b8377642c54550f0504f65688ba3a4967950cafe461df2"}], "totalBytes": 115515347}""")
TAPAS_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "tapas-large-wtq", "modelId": "google/tapas-large-finetuned-wtq", "revision": "f58317ab2577d17647d9acafa790c744a0388b30", "files": [{"path": "README.md", "bytes": 7224, "sha256": "2397bdae46316684903465e6f425a9fe10c85d491a9755ec7a31a74353c034c8"}, {"path": "config.json", "bytes": 1659, "sha256": "834f44a325a349d2d209ecb18e4fa2ca3cdb16cef5b169e5c0a8f5e16c444d13"}, {"path": "model.safetensors", "bytes": 1346985282, "sha256": "149247e13732c222ba621e0c4e7b90ba260869b36adcbe115dc872c2c98bccf0"}, {"path": "special_tokens_map.json", "bytes": 154, "sha256": "e3ec7abc6bcd45aba696cb95e9945186dad920aea78733f8b45c83d696ed0dea"}, {"path": "tokenizer_config.json", "bytes": 490, "sha256": "ab033df4f3c902cbc66bae2736bc99f4a59c43e7d9e53e40046576826b2cc5e9"}, {"path": "vocab.txt", "bytes": 262028, "sha256": "4d96f9308bcf9019684fcc109aa8c042b9b745edabba0162fbe66c75ebee2db4"}], "totalBytes": 1347256837}""")

DETECTION_DIR=Path("weights/table-transformer-detection")
STRUCTURE_DIR=Path("weights/table-transformer-structure-v1.1-all")
TAPAS_DIR=Path("weights/tapas-large-wtq")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""):
            h.update(chunk)
    return h.hexdigest()

def stage_snapshot(root,manifest):
    root.mkdir(parents=True,exist_ok=True)
    (root/"dimer-base-manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    for entry in manifest["files"]:
        path=root/entry["path"]
        if not path.is_file():
            hf_hub_download(
                repo_id=manifest["modelId"],
                filename=entry["path"],
                revision=manifest["revision"],
                local_dir=str(root),
            )
    for entry in manifest["files"]:
        path=root/entry["path"]
        if path.stat().st_size!=entry["bytes"]:
            raise RuntimeError(f"{manifest['modelId']} {entry['path']} size mismatch")
        if sha256_file(path)!=entry["sha256"]:
            raise RuntimeError(f"{manifest['modelId']} {entry['path']} SHA-256 mismatch")
    return {"model_id":manifest["modelId"],"revision":manifest["revision"],"bytes":manifest["totalBytes"]}

print(stage_snapshot(DETECTION_DIR,DETECTION_MANIFEST))
print(stage_snapshot(STRUCTURE_DIR,STRUCTURE_MANIFEST))
print(stage_snapshot(TAPAS_DIR,TAPAS_MANIFEST))

## 6. SciTSR-PD provenance

Pinned dataset:

`bevaya/SciTSR-pd @ dae336efa7af07d69194a510ff5568980e8ef253`

The dataset contains 108 public-domain scientific tables (89 train / 19 test) with:

- table image;
- logical cells;
- text chunks;
- paper identity;
- source license metadata.

The two source parquet files are verified by byte size and SHA-256 before they are read.

In [ ]:
SCITSR_REPO="bevaya/SciTSR-pd"
SCITSR_REVISION="dae336efa7af07d69194a510ff5568980e8ef253"
SCITSR_FILES=json.loads(r"""{"train": {"path": "data/train-00000-of-00001.parquet", "bytes": 5185141, "sha256": "d295b918777425a26fa13a5c3ba076517429682ede8c71b671a8b6115968e465", "rows": 89}, "test": {"path": "data/test-00000-of-00001.parquet", "bytes": 1066042, "sha256": "73af6317219349a08acdf1c8fee7cff21b463cdded64111b643123f7df4f8bed", "rows": 19}}""")
SCITSR_DIR=Path("weights/scitsr-pd")
SCITSR_DIR.mkdir(parents=True,exist_ok=True)

scitsr_paths={}
for split,spec in SCITSR_FILES.items():
    path=Path(hf_hub_download(
        repo_id=SCITSR_REPO,
        repo_type="dataset",
        filename=spec["path"],
        revision=SCITSR_REVISION,
        local_dir=str(SCITSR_DIR),
    ))
    if path.stat().st_size!=spec["bytes"]:
        raise RuntimeError(f"{split} SciTSR-PD shard size mismatch")
    digest=sha256_file(path)
    if digest!=spec["sha256"]:
        raise RuntimeError(f"{split} SciTSR-PD shard SHA-256 mismatch")
    scitsr_paths[split]=path
    print(split,path,digest)

## 7. Derive a table-geometry/text companion from SciTSR-PD

The source dataset stores logical cells and PDF-coordinate text chunks. The notebook derives a pixel-space companion:

- align chunk boxes to the rendered table image;
- match cell text to chunks;
- build pixel boxes for logical cells;
- derive row/column boxes;
- retain cell row/column spans and text.

This is a deterministic data transformation, not model inference.

Tables whose cell text cannot be aligned reliably are excluded from the geometry corpus and reported.

In [ ]:
from datasets import load_dataset
from collections import defaultdict
import io

data_files={k:str(v) for k,v in scitsr_paths.items()}
ds=load_dataset("parquet",data_files=data_files)

def norm_text(s):
    return re.sub(r"\s+"," ",re.sub(r"[^0-9A-Za-z]+"," ",str(s))).strip().lower()

def image_digest(image):
    rgb=image.convert("RGB")
    return hashlib.sha256(f"{rgb.width}x{rgb.height}:".encode()+rgb.tobytes()).hexdigest()

def dark_integral(image):
    g=np.asarray(image.convert("L"),dtype=np.uint8)
    mask=(g<210).astype(np.int32)
    return mask.cumsum(0).cumsum(1)

def rect_sum(ii,x0,y0,x1,y1):
    h,w=ii.shape
    x0=max(0,min(w-1,int(x0))); x1=max(0,min(w,int(x1)))
    y0=max(0,min(h-1,int(y0))); y1=max(0,min(h,int(y1)))
    if x1<=x0 or y1<=y0:
        return 0
    def at(y,x):
        if y<0 or x<0:return 0
        return int(ii[min(y,h-1),min(x,w-1)])
    return at(y1-1,x1-1)-at(y0-1,x1-1)-at(y1-1,x0-1)+at(y0-1,x0-1)

def align_chunks(image,chunks):
    scale=150.0/72.0
    h=image.height
    base=[]
    for c in chunks:
        x0=float(c["x1"])*scale
        x1=float(c["x2"])*scale
        y0=h-float(c["y2"])*scale
        y1=h-float(c["y1"])*scale
        base.append([x0,y0,x1,y1])
    ii=dark_integral(image)
    best=None
    # Most carrier tables use small positive offsets. Search a bounded integer grid.
    for dx in range(-8,33,2):
        for dy in range(-8,33,2):
            score=0
            area=0
            for x0,y0,x1,y1 in base:
                xx0,yy0,xx1,yy1=x0+dx,y0+dy,x1+dx,y1+dy
                score+=rect_sum(ii,xx0,yy0,xx1,yy1)
                area+=max(1,int((xx1-xx0)*(yy1-yy0)))
            value=score/max(1,area)
            if best is None or value>best[0]:
                best=(value,dx,dy)
    _,dx,dy=best
    boxes=[]
    for x0,y0,x1,y1 in base:
        boxes.append([
            max(0.0,x0+dx),max(0.0,y0+dy),
            min(float(image.width),x1+dx),min(float(image.height),y1+dy)
        ])
    return boxes,{"dx":dx,"dy":dy,"scale":scale,"ink_score":best[0]}

def match_cells_to_chunks(cells,chunks,chunk_boxes):
    chunk_norm=[norm_text(c.get("text","")) for c in chunks]
    used=set()
    out=[]
    for cell in sorted(cells,key=lambda c:(int(c["start_row"]),int(c["start_col"]),int(c.get("id",0)))):
        text=str(cell.get("tex") or " ".join(cell.get("content") or []))
        target=norm_text(text)
        if not target:
            continue
        exact=[i for i,t in enumerate(chunk_norm) if i not in used and t==target]
        matches=[]
        if exact:
            matches=[exact[0]]
        else:
            # Try short consecutive chunk runs in reading order.
            for i in range(len(chunks)):
                if i in used: continue
                parts=[]
                idxs=[]
                for j in range(i,min(len(chunks),i+5)):
                    if j in used: break
                    parts.append(chunk_norm[j]); idxs.append(j)
                    joined=norm_text(" ".join(parts))
                    if joined==target:
                        matches=idxs
                        break
                    if len(joined)>len(target)+10:
                        break
                if matches: break
        if not matches:
            return None
        for i in matches: used.add(i)
        b=[chunk_boxes[i] for i in matches]
        box=[min(x[0] for x in b),min(x[1] for x in b),max(x[2] for x in b),max(x[3] for x in b)]
        out.append({
            "text":text,
            "row_start":int(cell["start_row"]),"row_end":int(cell["end_row"]),
            "col_start":int(cell["start_col"]),"col_end":int(cell["end_col"]),
            "box":box,
        })
    return out

def decode_source_image(value):
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            image=Image.open(io.BytesIO(value["bytes"]))
            image.load()
            return image.convert("RGB")
        if value.get("path") and Path(value["path"]).is_file():
            image=Image.open(value["path"])
            image.load()
            return image.convert("RGB")
    if isinstance(value,(bytes,bytearray)):
        image=Image.open(io.BytesIO(value))
        image.load()
        return image.convert("RGB")
    raise TypeError(f"Unsupported SciTSR image representation: {type(value).__name__}")

def derive_table(row):
    image=decode_source_image(row["image"])
    chunk_boxes,alignment=align_chunks(image,row["chunks"])
    cell_records=match_cells_to_chunks(row["cells"],row["chunks"],chunk_boxes)
    if not cell_records:
        return None
    n_rows=1+max(c["row_end"] for c in cell_records)
    n_cols=1+max(c["col_end"] for c in cell_records)
    if n_rows<1 or n_cols<1:
        return None

    row_content=[]
    for r in range(n_rows):
        boxes=[c["box"] for c in cell_records if c["row_start"]<=r<=c["row_end"]]
        if not boxes:return None
        row_content.append([min(b[0] for b in boxes),min(b[1] for b in boxes),max(b[2] for b in boxes),max(b[3] for b in boxes)])
    col_content=[]
    for cidx in range(n_cols):
        boxes=[c["box"] for c in cell_records if c["col_start"]<=cidx<=c["col_end"]]
        if not boxes:return None
        col_content.append([min(b[0] for b in boxes),min(b[1] for b in boxes),max(b[2] for b in boxes),max(b[3] for b in boxes)])

    left=min(c["box"][0] for c in cell_records); right=max(c["box"][2] for c in cell_records)
    top=min(c["box"][1] for c in cell_records); bottom=max(c["box"][3] for c in cell_records)
    table_box=[max(0,left-2),max(0,top-2),min(image.width,right+2),min(image.height,bottom+2)]

    row_centers=[(b[1]+b[3])/2 for b in row_content]
    row_bounds=[table_box[1]]+[(row_centers[i]+row_centers[i+1])/2 for i in range(n_rows-1)]+[table_box[3]]
    rows=[[table_box[0],row_bounds[i],table_box[2],row_bounds[i+1]] for i in range(n_rows)]

    col_centers=[(b[0]+b[2])/2 for b in col_content]
    col_bounds=[table_box[0]]+[(col_centers[i]+col_centers[i+1])/2 for i in range(n_cols-1)]+[table_box[2]]
    cols=[[col_bounds[i],table_box[1],col_bounds[i+1],table_box[3]] for i in range(n_cols)]

    objects=[{"label":"table","box":table_box}]
    objects += [{"label":"table row","box":b} for b in rows]
    objects += [{"label":"table column","box":b} for b in cols]
    for c in cell_records:
        if c["row_start"]!=c["row_end"] or c["col_start"]!=c["col_end"]:
            objects.append({"label":"table spanning cell","box":c["box"]})

    grid=[["" for _ in range(n_cols)] for __ in range(n_rows)]
    for c in cell_records:
        if c["row_start"]==c["row_end"] and c["col_start"]==c["col_end"]:
            grid[c["row_start"]][c["col_start"]]=c["text"]

    return {
        "source_id":str(row["table_id"]),
        "paper_id":str(row["paper_id"]),
        "paper_title":str(row["paper_title"]),
        "paper_license":str(row["paper_license"]),
        "image":image,
        "pixel_sha256":image_digest(image),
        "cells":cell_records,
        "objects":objects,
        "n_rows":n_rows,
        "n_cols":n_cols,
        "grid":grid,
        "has_spans":any(c["row_start"]!=c["row_end"] or c["col_start"]!=c["col_end"] for c in cell_records),
        "alignment":alignment,
    }

derived=[]
failed=[]
for split in ("train","test"):
    for row in ds[split]:
        item=derive_table(row)
        if item is None:
            failed.append(str(row["table_id"]))
        else:
            derived.append(item)

print({
    "source_tables":sum(len(ds[s]) for s in ("train","test")),
    "derived_tables":len(derived),
    "excluded_alignment_or_mapping":len(failed),
})

## 8. Paper-disjoint split

The corpus is split by paper, not by individual table.

The canonical carrier target is approximately:

- 10 test papers
- 7 validation papers
- remaining papers for train

The exact resulting table counts are printed and recorded. A clean release run should compare them against the current carrier's 21 / 23 / 50 table split.

In [ ]:
rng=random.Random(42)
by_paper=defaultdict(list)
for item in derived:
    by_paper[item["paper_id"]].append(item)

papers=sorted(by_paper)
rng.shuffle(papers)
if len(papers)<18:
    raise RuntimeError("Too few derived papers after alignment")

test_papers=set(papers[:10])
val_papers=set(papers[10:17])
train_papers=set(papers[17:])

splits={
    "test":[x for p in test_papers for x in by_paper[p]],
    "validation":[x for p in val_papers for x in by_paper[p]],
    "train":[x for p in train_papers for x in by_paper[p]],
}
print({k:{"tables":len(v),"papers":len({x["paper_id"] for x in v})} for k,v in splits.items()})

## 9. Select ten QA-eligible rectangular tables

The full end-to-end QA workflow intentionally excludes spanning-cell tables.

Eligibility:

- no spanning cells;
- 3–20 rows;
- 2–12 columns;
- complete first row with unique non-empty headers;
- complete first body column with unique row identifiers;
- at least one numeric body column;
- cells ≤200 characters.

Selection is deterministic and independent of model output.

In [ ]:
NUMBER_RE=re.compile(r"^[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?%?$")

def parse_number(text):
    s=str(text).strip().replace(",","")
    if s.endswith("%"): s=s[:-1]
    if not NUMBER_RE.match(str(text).strip()):
        try:
            # permit comma-stripped representation
            float(s)
        except Exception:
            return None
    try:return float(s)
    except Exception:return None

def eligible(item):
    if item["has_spans"]: return False
    if not (3<=item["n_rows"]<=20 and 2<=item["n_cols"]<=12): return False
    grid=item["grid"]
    if any(not str(x).strip() or len(str(x))>200 for row in grid for x in row): return False
    headers=[norm_text(x) for x in grid[0]]
    if any(not x for x in headers) or len(set(headers))!=len(headers): return False
    row_ids=[norm_text(row[0]) for row in grid[1:]]
    if any(not x for x in row_ids) or len(set(row_ids))!=len(row_ids): return False
    numeric_cols=[]
    for c in range(1,item["n_cols"]):
        vals=[parse_number(row[c]) for row in grid[1:]]
        if len(vals)>=2 and all(v is not None for v in vals):
            numeric_cols.append(c)
    return bool(numeric_cols)

eligible_tables=[x for x in splits["test"] if eligible(x)]
eligible_tables.sort(key=lambda x:hashlib.sha256(f"42:{x['source_id']}".encode()).hexdigest())
if len(eligible_tables)<END_TO_END_TABLES:
    # Fall back to eligible tables from the whole paper-disjoint corpus, still chosen before inference.
    extra=[x for x in splits["validation"]+splits["train"] if eligible(x)]
    extra.sort(key=lambda x:hashlib.sha256(f"42:{x['source_id']}".encode()).hexdigest())
    eligible_tables += extra

canonical_tables=eligible_tables[:END_TO_END_TABLES]
if len(canonical_tables)!=END_TO_END_TABLES:
    raise RuntimeError(f"Only {len(canonical_tables)} QA-eligible tables were derived")

print([x["source_id"] for x in canonical_tables])

## 10. Create deterministic document pages

The real SciTSR table pixels are placed unchanged on a simple white page.

This creates measurable page-level table-detection ground truth without introducing a second table.

In [ ]:
FONT=ImageFont.load_default(size=22)

def make_page(item):
    table=item["image"]
    left,top,right,bottom=120,160,120,120
    page=Image.new("RGB",(table.width+left+right,table.height+top+bottom),"white")
    draw=ImageDraw.Draw(page)
    draw.text((left,48),f"Table {item['source_id']}",font=FONT,fill="black")
    page.paste(table,(left,top))
    draw.text((left,top+table.height+40),"Scientific table excerpt",font=FONT,fill="gray")
    gt_box=[left,top,left+table.width,top+table.height]
    return page,gt_box

for item in canonical_tables:
    item["page"],item["page_gt_box"]=make_page(item)
    item["page_sha256"]=image_digest(item["page"])

print({"pages":len(canonical_tables),"page_sizes":[x["page"].size for x in canonical_tables[:3]]})

## 11. Common detection/structure metrics

Bounding-box comparisons use pixel-space IoU.

AP is Pascal-VOC-style all-point AP over score-ranked predictions with one-to-one matching.

In [ ]:
def box_iou(a,b):
    x0=max(float(a[0]),float(b[0])); y0=max(float(a[1]),float(b[1]))
    x1=min(float(a[2]),float(b[2])); y1=min(float(a[3]),float(b[3]))
    inter=max(0,x1-x0)*max(0,y1-y0)
    aa=max(0,float(a[2])-float(a[0]))*max(0,float(a[3])-float(a[1]))
    bb=max(0,float(b[2])-float(b[0]))*max(0,float(b[3])-float(b[1]))
    u=aa+bb-inter
    return inter/u if u>0 else 0.0

def ap_for_label(preds,refs,label,thr):
    det=[]
    n_gt=0
    for i,(p,r) in enumerate(zip(preds,refs,strict=True)):
        gt=[x["box"] for x in r if x["label"]==label]
        n_gt+=len(gt)
        det += [(float(x["score"]),i,x["box"]) for x in p if x["label"]==label]
    det.sort(key=lambda x:-x[0])
    matched=defaultdict(set); hits=[]
    for score,i,box in det:
        gt=[x["box"] for x in refs[i] if x["label"]==label]
        best,bj=0,None
        for j,g in enumerate(gt):
            if j in matched[i]: continue
            v=box_iou(box,g)
            if v>best: best,bj=v,j
        if bj is not None and best>=thr:
            matched[i].add(bj); hits.append(1)
        else:hits.append(0)
    if not hits or n_gt==0:return 0.0
    tp=np.cumsum(hits); recall=tp/n_gt; precision=tp/np.arange(1,len(hits)+1)
    mr=np.concatenate([[0],recall,[1]]); mp=np.concatenate([[0],precision,[0]])
    for i in range(len(mp)-2,-1,-1):mp[i]=max(mp[i],mp[i+1])
    idx=np.where(mr[1:]!=mr[:-1])[0]
    return float(np.sum((mr[idx+1]-mr[idx])*mp[idx+1]))

## 12. Stage 1 — Table Transformer Detection

The detector runs on the synthetic page. The highest-scoring surviving `table` / `table rotated` detection becomes the target crop.

A miss is a real pipeline failure and propagates downstream.

In [ ]:
from transformers import AutoImageProcessor, TableTransformerForObjectDetection

t0=time.perf_counter()
det_processor=AutoImageProcessor.from_pretrained(str(DETECTION_DIR),local_files_only=True,trust_remote_code=False)
det_model=TableTransformerForObjectDetection.from_pretrained(
    str(DETECTION_DIR),
    local_files_only=True,
    trust_remote_code=False,
    use_pretrained_backbone=False,
).to(DEVICE).eval()
det_load_seconds=time.perf_counter()-t0
for p in det_model.parameters():p.requires_grad_(False)
det_params=sum(p.numel() for p in det_model.parameters())

def detect_table(image,threshold=DETECTION_THRESHOLD):
    inputs=det_processor(images=image,return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out=det_model(**inputs)
    result=det_processor.post_process_object_detection(
        out,target_sizes=[image.size[::-1]],threshold=float(threshold)
    )[0]
    rows=[]
    for score,label,box in zip(result["scores"],result["labels"],result["boxes"],strict=True):
        name=det_model.config.id2label[int(label)]
        if name in ("table","table rotated"):
            rows.append({"label":name,"score":float(score),"box":[float(v) for v in box.tolist()]})
    rows.sort(key=lambda x:-x["score"])
    return rows

_ = detect_table(canonical_tables[0]["page"])
if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
det_times=[]
for i,item in enumerate(canonical_tables):
    if torch.cuda.is_available():torch.cuda.synchronize()
    t=time.perf_counter(); preds=detect_table(item["page"])
    if torch.cuda.is_available():torch.cuda.synchronize()
    det_times.append(time.perf_counter()-t)
    item["detections"]=preds
    item["selected_detection"]=preds[0] if preds else None
det_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None

print({"mean_latency":float(np.mean(det_times)),"detections":[len(x["detections"]) for x in canonical_tables]})

## 13. Detection metrics

Each page has exactly one ground-truth table box.

The tutorial reports AP50/AP75, hit rate and best-box IoU across the ten pages.

In [ ]:
det_pred_lists=[]
det_refs=[]
det_rows=[]
for item in canonical_tables:
    preds=[{"label":"table","score":x["score"],"box":x["box"]} for x in item["detections"]]
    det_pred_lists.append(preds)
    det_refs.append([{"label":"table","box":item["page_gt_box"]}])
    best=max([box_iou(x["box"],item["page_gt_box"]) for x in item["detections"]],default=0.0)
    det_rows.append({
        "table_id":item["source_id"],"predicted":bool(item["detections"]),
        "best_iou":best,"hit50":best>=0.5,"hit75":best>=0.75,
    })

det_metrics={
    "ap50":ap_for_label(det_pred_lists,det_refs,"table",0.5),
    "ap75":ap_for_label(det_pred_lists,det_refs,"table",0.75),
    "hit50":float(np.mean([r["hit50"] for r in det_rows])),
    "hit75":float(np.mean([r["hit75"] for r in det_rows])),
    "mean_best_iou":float(np.mean([r["best_iou"] for r in det_rows])),
    "median_best_iou":float(np.median([r["best_iou"] for r in det_rows])),
}
print(det_metrics)

del det_model,det_processor
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

## 14. Build gold and detected crops

The structure recognizer is evaluated twice:

- **gold crop** — exact known table location, with 10 px page padding;
- **detected crop** — predicted box, with the same padding.

Ground-truth structure boxes are transformed into each crop's coordinate frame.

In [ ]:
def padded_crop(page,box,pad=CROP_PADDING):
    x0,y0,x1,y1=[float(v) for v in box]
    cx0=max(0,int(math.floor(x0-pad))); cy0=max(0,int(math.floor(y0-pad)))
    cx1=min(page.width,int(math.ceil(x1+pad))); cy1=min(page.height,int(math.ceil(y1+pad)))
    return page.crop((cx0,cy0,cx1,cy1)),[cx0,cy0,cx1,cy1]

def translate_gold_objects(item,crop_box):
    page_x0,page_y0=120,160
    cx0,cy0,cx1,cy1=crop_box
    out=[]
    for obj in item["objects"]:
        x0,y0,x1,y1=obj["box"]
        b=[x0+page_x0-cx0,y0+page_y0-cy0,x1+page_x0-cx0,y1+page_y0-cy0]
        b=[max(0,b[0]),max(0,b[1]),min(cx1-cx0,b[2]),min(cy1-cy0,b[3])]
        if b[2]>b[0] and b[3]>b[1]:
            out.append({"label":obj["label"],"box":b})
    return out

for item in canonical_tables:
    gold_crop,gold_crop_box=padded_crop(item["page"],item["page_gt_box"])
    item["gold_crop"]=gold_crop; item["gold_crop_box"]=gold_crop_box
    item["gold_crop_objects"]=translate_gold_objects(item,gold_crop_box)

    if item["selected_detection"] is not None:
        det_crop,det_crop_box=padded_crop(item["page"],item["selected_detection"]["box"])
        item["det_crop"]=det_crop; item["det_crop_box"]=det_crop_box
        item["det_crop_objects"]=translate_gold_objects(item,det_crop_box)
    else:
        item["det_crop"]=None; item["det_crop_box"]=None; item["det_crop_objects"]=[]

## 15. Stage 2 — Table Transformer Structure Recognition

Raw model output is preserved.

Rows and columns are later subjected to deterministic same-label NMS for grid reconstruction. NMS is a workshop composition rule, not part of the model.

In [ ]:
struct_processor=AutoImageProcessor.from_pretrained(
    str(STRUCTURE_DIR),
    local_files_only=True,
    trust_remote_code=False,
    size={"shortest_edge":800,"longest_edge":800},
)
t0=time.perf_counter()
struct_model=TableTransformerForObjectDetection.from_pretrained(
    str(STRUCTURE_DIR),
    local_files_only=True,
    trust_remote_code=False,
    use_pretrained_backbone=False,
).to(DEVICE).eval()
struct_load_seconds=time.perf_counter()-t0
for p in struct_model.parameters():p.requires_grad_(False)
struct_params=sum(p.numel() for p in struct_model.parameters())

def recognize_structure(image,threshold=STRUCTURE_THRESHOLD):
    inputs=struct_processor(images=image,return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out=struct_model(**inputs)
    result=struct_processor.post_process_object_detection(
        out,target_sizes=[image.size[::-1]],threshold=float(threshold)
    )[0]
    rows=[]
    for score,label,box in zip(result["scores"],result["labels"],result["boxes"],strict=True):
        name=struct_model.config.id2label[int(label)]
        rows.append({"label":name,"score":float(score),"box":[float(v) for v in box.tolist()]})
    rows.sort(key=lambda x:-x["score"])
    return rows

_ = recognize_structure(canonical_tables[0]["gold_crop"])
if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
struct_times=[]
for item in canonical_tables:
    t=time.perf_counter()
    item["gold_structure_raw"]=recognize_structure(item["gold_crop"])
    if item["det_crop"] is not None:
        item["det_structure_raw"]=recognize_structure(item["det_crop"])
    else:
        item["det_structure_raw"]=[]
    struct_times.append(time.perf_counter()-t)
struct_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
print({"mean_pair_seconds":float(np.mean(struct_times))})

## 16. Grid reconstruction

Rows are sorted top-to-bottom; columns left-to-right.

Same-label duplicate boxes are removed with NMS at IoU 0.50 before grid construction.

In [ ]:
def nms_items(items,iou_threshold=GRID_NMS_IOU):
    kept=[]
    for item in sorted(items,key=lambda x:-x["score"]):
        if all(box_iou(item["box"],k["box"])<iou_threshold for k in kept):
            kept.append(item)
    return kept

def grid_from_structure(raw):
    rows=nms_items([x for x in raw if x["label"]=="table row"])
    cols=nms_items([x for x in raw if x["label"]=="table column"])
    rows.sort(key=lambda x:(x["box"][1]+x["box"][3])/2)
    cols.sort(key=lambda x:(x["box"][0]+x["box"][2])/2)
    cells=[]
    valid=bool(rows and cols)
    for ri,r in enumerate(rows):
        row=[]
        for ci,c in enumerate(cols):
            b=[
                max(r["box"][0],c["box"][0]),max(r["box"][1],c["box"][1]),
                min(r["box"][2],c["box"][2]),min(r["box"][3],c["box"][3]),
            ]
            if b[2]<=b[0] or b[3]<=b[1]:valid=False
            row.append(b)
        cells.append(row)
    return {"rows":rows,"columns":cols,"cells":cells,"valid_geometry":valid}

for item in canonical_tables:
    item["gold_grid_pred"]=grid_from_structure(item["gold_structure_raw"])
    item["det_grid_pred"]=grid_from_structure(item["det_structure_raw"]) if item["det_crop"] is not None else {"rows":[],"columns":[],"cells":[],"valid_geometry":False}

## 17. Assign SciTSR source cell text to predicted cells

For each gold logical cell, its pixel center is transformed into the current crop and located in the predicted row and column.

No gold row/column index is supplied to the reconstruction algorithm.

In [ ]:
def source_cells_in_crop(item,crop_box):
    cx0,cy0,_,_=crop_box
    page_left,page_top=120,160
    out=[]
    for c in item["cells"]:
        b=c["box"]
        out.append({
            **c,
            "crop_box":[b[0]+page_left-cx0,b[1]+page_top-cy0,b[2]+page_left-cx0,b[3]+page_top-cy0],
        })
    return out

def contains(box,x,y):
    return box[0]<=x<=box[2] and box[1]<=y<=box[3]

def reconstruct_table(item,grid,crop_box):
    if not grid["valid_geometry"] or not grid["rows"] or not grid["columns"]:
        return {"valid":False,"reason":"grid_invalid","table":None,"mapped":0,"unmapped":len(item["cells"]),"ambiguous":0}
    strings=[[[] for _ in grid["columns"]] for __ in grid["rows"]]
    mapped=unmapped=ambiguous=correct=0
    simple=[c for c in source_cells_in_crop(item,crop_box) if c["row_start"]==c["row_end"] and c["col_start"]==c["col_end"]]
    for c in simple:
        b=c["crop_box"]; x=(b[0]+b[2])/2; y=(b[1]+b[3])/2
        ris=[i for i,r in enumerate(grid["rows"]) if contains(r["box"],x,y)]
        cis=[i for i,col in enumerate(grid["columns"]) if contains(col["box"],x,y)]
        if len(ris)==1 and len(cis)==1:
            ri,ci=ris[0],cis[0]
            strings[ri][ci].append(c["text"])
            mapped+=1
            if ri==c["row_start"] and ci==c["col_start"]:correct+=1
        elif len(ris)==0 or len(cis)==0:
            unmapped+=1
        else:
            ambiguous+=1
    table=[[" ".join(cell).strip() for cell in row] for row in strings]
    headers=table[0] if table else []
    norm_headers=[norm_text(x) for x in headers]
    valid_headers=bool(headers) and all(norm_headers) and len(set(norm_headers))==len(norm_headers)
    valid=grid["valid_geometry"] and valid_headers and len(table)<=64 and len(headers)<=32
    return {
        "valid":valid,
        "reason":"ok" if valid else "header_or_limit_invalid",
        "table":table,
        "mapped":mapped,"unmapped":unmapped,"ambiguous":ambiguous,
        "assignment_accuracy":correct/max(1,len(simple)),
        "headers_valid":valid_headers,
    }

for item in canonical_tables:
    item["gold_recon"]=reconstruct_table(item,item["gold_grid_pred"],item["gold_crop_box"])
    if item["det_crop"] is not None:
        item["det_recon"]=reconstruct_table(item,item["det_grid_pred"],item["det_crop_box"])
    else:
        item["det_recon"]={"valid":False,"reason":"detection_miss","table":None,"mapped":0,"unmapped":len(item["cells"]),"ambiguous":0}

## 18. Structure and reconstruction metrics

Rows/columns are scored independently on gold crops and detected crops.

Header predictions are not penalized because SciTSR-PD does not annotate the two PubTables header classes used by the structure checkpoint.

In [ ]:
def refs_for(item,key,label):
    return [x for x in item[key] if x["label"]==label]

def preds_for(item,key,label):
    return [x for x in item[key] if x["label"]==label]

structure_rows=[]
for path_name,pred_key,ref_key in [
    ("gold_crop","gold_structure_raw","gold_crop_objects"),
    ("detected_crop","det_structure_raw","det_crop_objects"),
]:
    for item in canonical_tables:
        gp=item["gold_grid_pred"] if path_name=="gold_crop" else item["det_grid_pred"]
        gold_rows=refs_for(item,ref_key,"table row")
        gold_cols=refs_for(item,ref_key,"table column")
        pred_rows=preds_for(item,pred_key,"table row")
        pred_cols=preds_for(item,pred_key,"table column")
        best_r=[max([box_iou(g["box"],p["box"]) for p in pred_rows],default=0) for g in gold_rows]
        best_c=[max([box_iou(g["box"],p["box"]) for p in pred_cols],default=0) for g in gold_cols]
        structure_rows.append({
            "table_id":item["source_id"],"path":path_name,
            "gold_rows":len(gold_rows),"predicted_rows":len(gp["rows"]),
            "gold_columns":len(gold_cols),"predicted_columns":len(gp["columns"]),
            "row_count_exact":len(gp["rows"])==len(gold_rows),
            "column_count_exact":len(gp["columns"])==len(gold_cols),
            "grid_shape_exact":len(gp["rows"])==len(gold_rows) and len(gp["columns"])==len(gold_cols),
            "mean_row_iou":float(np.mean(best_r)) if best_r else 0.0,
            "mean_column_iou":float(np.mean(best_c)) if best_c else 0.0,
        })

for path_name in ("gold_crop","detected_crop"):
    subset=[r for r in structure_rows if r["path"]==path_name]
    print(path_name,{
        "grid_exact":float(np.mean([r["grid_shape_exact"] for r in subset])),
        "mean_row_iou":float(np.mean([r["mean_row_iou"] for r in subset])),
        "mean_column_iou":float(np.mean([r["mean_column_iou"] for r in subset])),
    })

del struct_model,struct_processor
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

## 19. Complex spanning-cell probe

A few held-out spanning-cell tables are displayed as structure-only examples.

They are intentionally excluded from rectangular TAPAS reconstruction because simple row × column intersections cannot faithfully represent their semantics.

In [ ]:
complex_tables=[x for x in splits["test"] if x["has_spans"]][:COMPLEX_STRUCTURE_TABLES]
print({"complex_probe_tables":[x["source_id"] for x in complex_tables]})

## 20. Generate 50 table-QA questions from the gold logical tables

Each table contributes:

- 2 lookup (`NONE`) questions;
- 1 `SUM`;
- 1 `AVERAGE`;
- 1 `COUNT`.

Generation uses only the gold logical table and happens independently of model predictions.

In [ ]:
def gold_table_dict(item):
    headers=item["grid"][0]
    body=item["grid"][1:]
    return {headers[c]:[row[c] for row in body] for c in range(len(headers))}

def numeric_columns(item):
    out=[]
    for c in range(1,item["n_cols"]):
        vals=[parse_number(row[c]) for row in item["grid"][1:]]
        if len(vals)>=2 and all(v is not None for v in vals):
            out.append((c,vals))
    return out

qa_records=[]
for item in canonical_tables:
    grid=item["grid"]; headers=grid[0]; body=grid[1:]
    row_ids=[row[0] for row in body]
    candidates=[]
    for r in range(len(body)):
        for c in range(1,len(headers)):
            candidates.append((r,c))
    candidates.sort(key=lambda rc:hashlib.sha256(f"{item['source_id']}:{rc[0]}:{rc[1]}".encode()).hexdigest())
    for j,(r,c) in enumerate(candidates[:2]):
        qa_records.append({
            "id":f"{item['source_id']}-lookup-{j}",
            "table_id":item["source_id"],
            "question":f"What is the {headers[c]} for {row_ids[r]}?",
            "category":"NONE",
            "gold_aggregation":"NONE",
            "gold_coordinates":[[r,c]],
            "gold_denotation":[body[r][c]],
        })
    c,vals=numeric_columns(item)[0]
    coords=[[r,c] for r in range(len(body))]
    qa_records += [
        {
            "id":f"{item['source_id']}-sum","table_id":item["source_id"],
            "question":f"What is the total {headers[c]}?","category":"SUM",
            "gold_aggregation":"SUM","gold_coordinates":coords,
            "gold_denotation":float(sum(vals)),
        },
        {
            "id":f"{item['source_id']}-avg","table_id":item["source_id"],
            "question":f"What is the average {headers[c]}?","category":"AVERAGE",
            "gold_aggregation":"AVERAGE","gold_coordinates":coords,
            "gold_denotation":float(sum(vals)/len(vals)),
        },
        {
            "id":f"{item['source_id']}-count","table_id":item["source_id"],
            "question":f"How many {headers[c]} entries are listed?","category":"COUNT",
            "gold_aggregation":"COUNT","gold_coordinates":coords,
            "gold_denotation":float(len(vals)),
        },
    ]

if len(qa_records)!=50:
    raise RuntimeError(f"Expected 50 questions, got {len(qa_records)}")
counts=defaultdict(int)
for r in qa_records:counts[r["category"]]+=1
if dict(counts)!={"NONE":20,"SUM":10,"AVERAGE":10,"COUNT":10}:
    raise RuntimeError(f"Question mix mismatch: {dict(counts)}")
print(dict(counts))

## 21. Stage 3 — TAPAS

TAPAS receives a string table and a question. It selects cells at threshold 0.5 and predicts one aggregation operator.

For `SUM`, `AVERAGE` and `COUNT`, the notebook computes the numeric answer from selected cells exactly as the DIMER reference carrier does.

In [ ]:
from transformers import TapasTokenizer, TapasForQuestionAnswering

tapas_tokenizer=TapasTokenizer.from_pretrained(str(TAPAS_DIR),local_files_only=True)
t0=time.perf_counter()
tapas_model=TapasForQuestionAnswering.from_pretrained(
    str(TAPAS_DIR),local_files_only=True,dtype=torch.float32
).to(DEVICE).eval()
tapas_load_seconds=time.perf_counter()-t0
for p in tapas_model.parameters():p.requires_grad_(False)
tapas_params=sum(p.numel() for p in tapas_model.parameters())
AGGREGATIONS=("NONE","SUM","AVERAGE","COUNT")

def compute_numeric(cells,aggregation):
    vals=[parse_number(x) for x in cells]
    if aggregation=="COUNT":return float(len(cells))
    if any(v is None for v in vals):return None
    if aggregation=="SUM":return float(sum(vals))
    if aggregation=="AVERAGE":return float(sum(vals)/len(vals)) if vals else None
    return None

def table_to_df(table):
    headers=table[0]
    body=table[1:]
    return pd.DataFrame(body,columns=headers,dtype=object)

def tapas_answer(table,question):
    df=table_to_df(table)
    inputs=tapas_tokenizer(
        table=df,
        queries=[question],
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    model_inputs={k:v.to(DEVICE) for k,v in inputs.items()}
    with torch.inference_mode():
        out=tapas_model(**model_inputs)
    coords,agg=tapas_tokenizer.convert_logits_to_predictions(
        inputs,
        out.logits.detach().cpu(),
        out.logits_aggregation.detach().cpu(),
    )
    selected=[list(x) for x in coords[0]]
    aggregation=AGGREGATIONS[int(agg[0])]
    cells=[str(df.iat[r,c]) for r,c in selected]
    numeric=compute_numeric(cells,aggregation)
    answer=(f"{aggregation} > " if aggregation!="NONE" else "")+", ".join(cells)
    return {
        "coordinates":selected,"cells":cells,"aggregation":aggregation,
        "numeric_answer":numeric,"answer":answer,
    }

print({"parameters":tapas_params,"load_seconds":tapas_load_seconds})

## 22. QA scoring

Denotation correctness is the headline downstream metric:

- lookup: same multiset of normalized selected cell strings;
- numeric aggregation: numeric answer matches within `1e-6`.

Gold-table evaluation also reports aggregation and exact selected-coordinate accuracy.

In [ ]:
def norm_cell(s):
    return " ".join(str(s).lower().split())

def denotation_correct(result,record):
    gold=record["gold_denotation"]
    if isinstance(gold,list):
        return sorted(norm_cell(x) for x in result["cells"])==sorted(norm_cell(x) for x in gold)
    value=result["numeric_answer"]
    return value is not None and abs(float(value)-float(gold))<=1e-6

def score_path(path_name,table_provider):
    rows=[]
    times=[]
    for record in qa_records:
        item=next(x for x in canonical_tables if x["source_id"]==record["table_id"])
        table,status=table_provider(item)
        if table is None:
            rows.append({
                "question_id":record["id"],"table_id":record["table_id"],"path":path_name,
                "question":record["question"],"gold_aggregation":record["gold_aggregation"],
                "predicted_aggregation":None,"gold_denotation":record["gold_denotation"],
                "predicted_denotation":None,"denotation_correct":False,
                "aggregation_correct":False,"selected_cells":[],"coordinates":[],
                "pipeline_status":status,"latency_seconds":None,
            })
            continue
        t=time.perf_counter()
        result=tapas_answer(table,record["question"])
        times.append(time.perf_counter()-t)
        rows.append({
            "question_id":record["id"],"table_id":record["table_id"],"path":path_name,
            "question":record["question"],"gold_aggregation":record["gold_aggregation"],
            "predicted_aggregation":result["aggregation"],"gold_denotation":record["gold_denotation"],
            "predicted_denotation":result["numeric_answer"] if record["gold_aggregation"]!="NONE" else result["cells"],
            "denotation_correct":denotation_correct(result,record),
            "aggregation_correct":result["aggregation"]==record["gold_aggregation"],
            "selected_cells":result["cells"],"coordinates":result["coordinates"],
            "pipeline_status":"ok","latency_seconds":times[-1],
        })
    return rows,times

def gold_provider(item):
    return item["grid"],"ok"

def struct_provider(item):
    r=item["gold_recon"]
    return (r["table"],"ok") if r["valid"] else (None,r["reason"])

def full_provider(item):
    if item["selected_detection"] is None:return None,"detection_miss"
    r=item["det_recon"]
    return (r["table"],"ok") if r["valid"] else (None,r["reason"])

gold_qa,gold_tapas_times=score_path("gold_table",gold_provider)
structure_qa,structure_tapas_times=score_path("structure_only",struct_provider)
end_to_end_qa,end_tapas_times=score_path("end_to_end",full_provider)

def qa_summary(rows):
    return {
        "n":len(rows),
        "denotation_accuracy":float(np.mean([r["denotation_correct"] for r in rows])),
        "aggregation_accuracy":float(np.mean([r["aggregation_correct"] for r in rows])),
        "valid_questions":sum(r["pipeline_status"]=="ok" for r in rows),
    }

print("Gold:",qa_summary(gold_qa))
print("Structure-only:",qa_summary(structure_qa))
print("End-to-end:",qa_summary(end_to_end_qa))

## 23. Pipeline waterfall

The waterfall counts failures rather than dropping them from downstream denominators.

In [ ]:
waterfall={
    "tables":len(canonical_tables),
    "detection":{
        "hit50":sum(r["hit50"] for r in det_rows),
        "hit75":sum(r["hit75"] for r in det_rows),
    },
    "structure_gold_crop":{
        "grid_exact":sum(r["grid_shape_exact"] for r in structure_rows if r["path"]=="gold_crop"),
    },
    "structure_detected_crop":{
        "grid_exact":sum(r["grid_shape_exact"] for r in structure_rows if r["path"]=="detected_crop"),
    },
    "reconstruction":{
        "valid_gold_crop":sum(x["gold_recon"]["valid"] for x in canonical_tables),
        "valid_detected_crop":sum(x["det_recon"]["valid"] for x in canonical_tables),
    },
    "qa":{
        "questions":50,
        "gold_table_denotation_accuracy":qa_summary(gold_qa)["denotation_accuracy"],
        "structure_only_denotation_accuracy":qa_summary(structure_qa)["denotation_accuracy"],
        "end_to_end_denotation_accuracy":qa_summary(end_to_end_qa)["denotation_accuracy"],
    },
}
print(json.dumps(waterfall,indent=2))

## 24. Workshop exercise

Before inspecting example panels, consider:

1. Can correct table detection still lead to wrong QA?
2. What happens if one row is missing?
3. What happens if the first reconstructed row is not a valid header?
4. Why can TAPAS not consume row/column boxes directly?
5. Where would OCR enter a production pipeline?
6. Which upstream stage appears to dominate the measured loss in the waterfall?

In [ ]:
for item in canonical_tables:
    print({
        "table":item["source_id"],
        "detection_iou":next(r["best_iou"] for r in det_rows if r["table_id"]==item["source_id"]),
        "gold_crop_valid":item["gold_recon"]["valid"],
        "detected_crop_valid":item["det_recon"]["valid"],
        "gold_shape":[item["n_rows"],item["n_cols"]],
        "gold_crop_pred_shape":[len(item["gold_grid_pred"]["rows"]),len(item["gold_grid_pred"]["columns"])],
        "det_crop_pred_shape":[len(item["det_grid_pred"]["rows"]),len(item["det_grid_pred"]["columns"])],
    })

## 25. Resource comparison

Models are loaded sequentially. The table records model size, load time, stage latency and peak accelerator memory where available.

In [ ]:
def median(x):return float(np.median(x)) if x else None
resource_rows=[
    {
        "stage":"detection","model":"Table Transformer Detection",
        "model_id":DETECTION_MANIFEST["modelId"],"revision":DETECTION_MANIFEST["revision"],
        "parameter_count":det_params,
        "weight_bytes":next(x["bytes"] for x in DETECTION_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "load_seconds":det_load_seconds,"mean_latency_seconds":float(np.mean(det_times)),
        "median_latency_seconds":median(det_times),"peak_gpu_memory_bytes":det_peak,
    },
    {
        "stage":"structure","model":"Table Transformer Structure",
        "model_id":STRUCTURE_MANIFEST["modelId"],"revision":STRUCTURE_MANIFEST["revision"],
        "parameter_count":struct_params,
        "weight_bytes":next(x["bytes"] for x in STRUCTURE_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "load_seconds":struct_load_seconds,"mean_latency_seconds":float(np.mean(struct_times)),
        "median_latency_seconds":median(struct_times),"peak_gpu_memory_bytes":struct_peak,
    },
    {
        "stage":"qa","model":"TAPAS Large WTQ",
        "model_id":TAPAS_MANIFEST["modelId"],"revision":TAPAS_MANIFEST["revision"],
        "parameter_count":tapas_params,
        "weight_bytes":next(x["bytes"] for x in TAPAS_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "load_seconds":tapas_load_seconds,
        "mean_latency_seconds":float(np.mean(gold_tapas_times+structure_tapas_times+end_tapas_times)),
        "median_latency_seconds":median(gold_tapas_times+structure_tapas_times+end_tapas_times),
        "peak_gpu_memory_bytes":torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
    },
]
for row in resource_rows:print(row)

## 26. Qualitative four-stage panels

Each panel shows:

1. synthetic document page + predicted table box;
2. detected crop + row/column structure;
3. reconstructed table text;
4. one representative TAPAS result.

Visualizations supplement machine-readable outputs.

In [ ]:
examples_dir=Path(OUTPUT_DIR)/"examples"
examples_dir.mkdir(parents=True,exist_ok=True)

def draw_boxes(image,items):
    out=image.convert("RGB").copy()
    d=ImageDraw.Draw(out)
    for item in items:
        d.rectangle(item["box"],width=2)
        d.text((item["box"][0]+2,item["box"][1]+2),item["label"],fill="black")
    return out

for item in canonical_tables[:4]:
    page=draw_boxes(item["page"],item["detections"][:3])
    crop=item["det_crop"] if item["det_crop"] is not None else item["gold_crop"]
    struct=item["det_structure_raw"] if item["det_crop"] is not None else item["gold_structure_raw"]
    crop_vis=draw_boxes(crop,[x for x in struct if x["label"] in ("table row","table column")])
    page.thumbnail((700,700)); crop_vis.thumbnail((700,700))
    canvas=Image.new("RGB",(max(page.width,crop_vis.width),page.height+crop_vis.height+120),"white")
    canvas.paste(page,(0,0)); canvas.paste(crop_vis,(0,page.height))
    q=next(r for r in end_to_end_qa if r["table_id"]==item["source_id"])
    d=ImageDraw.Draw(canvas)
    d.text((5,page.height+crop_vis.height+8),f"Q: {q['question']}",fill="black")
    d.text((5,page.height+crop_vis.height+38),f"Gold: {q['gold_denotation']}",fill="black")
    d.text((5,page.height+crop_vis.height+68),f"Path status: {q['pipeline_status']}  Pred: {q['predicted_denotation']}",fill="black")
    path=examples_dir/f"{item['source_id'].replace('/','_')}.png"
    canvas.save(path)
    print(path)

## 27. Machine-readable exports

Outputs include stage metrics, reconstructed tables, QA records, waterfall, resource metrics and provenance.

In [ ]:
import csv
from datetime import datetime, timezone

out_dir=Path(OUTPUT_DIR)
out_dir.mkdir(parents=True,exist_ok=True)
tables_dir=out_dir/"tables"
tables_dir.mkdir(exist_ok=True)

def write_csv(path,rows,fieldnames):
    with open(path,"w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=fieldnames); w.writeheader()
        for row in rows:
            out={}
            for k in fieldnames:
                v=row.get(k)
                if isinstance(v,(list,dict)):v=json.dumps(v)
                out[k]=v
            w.writerow(out)

write_csv(out_dir/"table_detection.csv",det_rows,["table_id","predicted","best_iou","hit50","hit75"])
write_csv(
    out_dir/"structure_metrics.csv",structure_rows,
    ["table_id","path","gold_rows","predicted_rows","gold_columns","predicted_columns",
     "row_count_exact","column_count_exact","grid_shape_exact","mean_row_iou","mean_column_iou"]
)

recon_rows=[]
for item in canonical_tables:
    for path_name,key in [("gold_crop","gold_recon"),("detected_crop","det_recon")]:
        r=item[key]
        recon_rows.append({
            "table_id":item["source_id"],"path":path_name,
            "gold_rows":item["n_rows"],"gold_columns":item["n_cols"],
            "predicted_rows":len(item["gold_grid_pred"]["rows"] if path_name=="gold_crop" else item["det_grid_pred"]["rows"]),
            "predicted_columns":len(item["gold_grid_pred"]["columns"] if path_name=="gold_crop" else item["det_grid_pred"]["columns"]),
            "mapped_cells":r.get("mapped",0),"unmapped_cells":r.get("unmapped",0),
            "ambiguous_cells":r.get("ambiguous",0),"cell_assignment_accuracy":r.get("assignment_accuracy"),
            "header_exact":r.get("headers_valid",False),"valid_for_tapas":r.get("valid",False),
        })
        if r.get("table") is not None:
            (tables_dir/f"{item['source_id'].replace('/','_')}_{path_name}.json").write_text(
                json.dumps({"table":r["table"],"findings":r["reason"]},indent=2),encoding="utf-8"
            )
    (tables_dir/f"{item['source_id'].replace('/','_')}_gold.json").write_text(
        json.dumps({"table":item["grid"]},indent=2),encoding="utf-8"
    )
write_csv(
    out_dir/"reconstruction_metrics.csv",recon_rows,
    ["table_id","path","gold_rows","gold_columns","predicted_rows","predicted_columns","mapped_cells",
     "unmapped_cells","ambiguous_cells","cell_assignment_accuracy","header_exact","valid_for_tapas"]
)

with open(out_dir/"qa_questions.jsonl","w",encoding="utf-8") as f:
    for r in qa_records:f.write(json.dumps(r,ensure_ascii=False)+"\n")

qa_all=gold_qa+structure_qa+end_to_end_qa
write_csv(
    out_dir/"qa_predictions.csv",qa_all,
    ["question_id","table_id","path","question","gold_aggregation","predicted_aggregation",
     "gold_denotation","predicted_denotation","denotation_correct","aggregation_correct",
     "selected_cells","coordinates","pipeline_status","latency_seconds"]
)
write_csv(
    out_dir/"resource_metrics.csv",resource_rows,
    ["stage","model","model_id","revision","parameter_count","weight_bytes","load_seconds",
     "mean_latency_seconds","median_latency_seconds","peak_gpu_memory_bytes"]
)
(out_dir/"waterfall.json").write_text(json.dumps(waterfall,indent=2),encoding="utf-8")

input_manifest={
    "source_tables":sum(len(ds[s]) for s in ("train","test")),
    "derived_tables":len(derived),
    "paper_split":{k:{"tables":len(v),"papers":len({x["paper_id"] for x in v})} for k,v in splits.items()},
    "canonical_table_ids":[x["source_id"] for x in canonical_tables],
    "complex_probe_ids":[x["source_id"] for x in complex_tables],
}
(out_dir/"input_manifest.json").write_text(json.dumps(input_manifest,indent=2),encoding="utf-8")

provenance={
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "notebook_spec":"2.1","profile":"TASK-INFERENCE","pedagogical_mode":"WORKSHOP",
    "dataset":{
        "id":SCITSR_REPO,"revision":SCITSR_REVISION,"license":"CC0-1.0",
        "source_files":SCITSR_FILES,"split_seed":42,
        "derivation":"runtime reconstruction of pixel geometry from SciTSR logical cells and text chunks",
    },
    "models":{
        "detection":{"id":DETECTION_MANIFEST["modelId"],"revision":DETECTION_MANIFEST["revision"]},
        "structure":{"id":STRUCTURE_MANIFEST["modelId"],"revision":STRUCTURE_MANIFEST["revision"]},
        "tapas":{"id":TAPAS_MANIFEST["modelId"],"revision":TAPAS_MANIFEST["revision"]},
    },
    "thresholds":{
        "detection":DETECTION_THRESHOLD,"structure":STRUCTURE_THRESHOLD,"grid_nms_iou":GRID_NMS_IOU
    },
    "runtime":RUNTIME,
}
(out_dir/"provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")

print("Outputs:")
for p in sorted(out_dir.iterdir()):print(" ",p)

## 28. BYOD (optional)

A production table-intelligence workflow needs text to populate predicted cells.

BYOD therefore requires:

- page image;
- OCR/text boxes;
- optional QA questions.

This notebook does not run OCR itself.

Recommended structure:

```text
dataset/
  pages/
    page001.png
  ocr.csv
  questions.csv   # optional
```

`ocr.csv`:

```text
page_id,text,x0,y0,x1,y1,order
```

OCR words would be assigned to predicted cells by word-center containment and joined in supplied reading order.

### Privacy

Tables may contain financial, personal, employee, medical or proprietary information. Do not upload restricted or regulated documents to a hosted runtime unless authorized.

In [ ]:
import csv

byod_result=None
if USE_BYOD:
    root=Path(BYOD_PATH)
    pages_dir=root/"pages"
    ocr_path=root/"ocr.csv"
    questions_path=root/"questions.csv"
    if not pages_dir.is_dir() or not ocr_path.is_file():
        raise FileNotFoundError("BYOD_PATH must contain pages/ and ocr.csv")

    with open(ocr_path,encoding="utf-8",newline="") as f:
        ocr_rows=list(csv.DictReader(f))
    required={"page_id","text","x0","y0","x1","y1","order"}
    if not ocr_rows or not required.issubset(ocr_rows[0]):
        raise ValueError(f"ocr.csv must contain {sorted(required)}")

    ocr_by_page=defaultdict(list)
    for row in ocr_rows:
        ocr_by_page[row["page_id"]].append({
            "text":row["text"],
            "box":[float(row[k]) for k in ("x0","y0","x1","y1")],
            "order":int(row["order"]),
        })
    for page_id in ocr_by_page:
        ocr_by_page[page_id].sort(key=lambda x:x["order"])

    q_by_page=defaultdict(list)
    if questions_path.is_file():
        with open(questions_path,encoding="utf-8",newline="") as f:
            question_rows=list(csv.DictReader(f))
        if question_rows and not {"id","page_id","question"}.issubset(question_rows[0]):
            raise ValueError("questions.csv must contain id,page_id,question; accepted_answer is optional")
        for row in question_rows:
            q_by_page[row["page_id"]].append(row)

    # Reload the two vision stages sequentially for BYOD.
    dp=AutoImageProcessor.from_pretrained(str(DETECTION_DIR),local_files_only=True,trust_remote_code=False)
    dm=TableTransformerForObjectDetection.from_pretrained(
        str(DETECTION_DIR),local_files_only=True,trust_remote_code=False,use_pretrained_backbone=False
    ).to(DEVICE).eval()
    det_processor,det_model=dp,dm

    byod_pages=[]
    for image_path in sorted(pages_dir.iterdir()):
        if image_path.suffix.lower() not in {".png",".jpg",".jpeg",".webp",".tif",".tiff",".bmp"}:
            continue
        page_id=image_path.stem
        if page_id not in ocr_by_page:
            continue
        image=Image.open(image_path); image.load(); image=image.convert("RGB")
        if min(image.size)<16 or max(image.size)>4096:
            raise ValueError(f"{image_path.name}: page side outside 16..4096")
        detections=detect_table(image)
        selected=detections[0] if detections else None
        byod_pages.append({"page_id":page_id,"image":image,"detection":selected})

    del det_model,det_processor
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

    sp=AutoImageProcessor.from_pretrained(
        str(STRUCTURE_DIR),local_files_only=True,trust_remote_code=False,
        size={"shortest_edge":800,"longest_edge":800},
    )
    sm=TableTransformerForObjectDetection.from_pretrained(
        str(STRUCTURE_DIR),local_files_only=True,trust_remote_code=False,use_pretrained_backbone=False
    ).to(DEVICE).eval()
    struct_processor,struct_model=sp,sm

    byod_outputs=[]
    for page in byod_pages:
        if page["detection"] is None:
            byod_outputs.append({"page_id":page["page_id"],"status":"detection_miss","table":None,"answers":[]})
            continue
        crop,crop_box=padded_crop(page["image"],page["detection"]["box"])
        raw=recognize_structure(crop)
        grid=grid_from_structure(raw)
        if not grid["valid_geometry"] or not grid["rows"] or not grid["columns"]:
            byod_outputs.append({"page_id":page["page_id"],"status":"grid_invalid","table":None,"answers":[]})
            continue

        # Assign OCR word centers into predicted row/column cells.
        strings=[[[] for _ in grid["columns"]] for __ in grid["rows"]]
        cx0,cy0,_,_=crop_box
        for word in ocr_by_page[page["page_id"]]:
            x=(word["box"][0]+word["box"][2])/2-cx0
            y=(word["box"][1]+word["box"][3])/2-cy0
            ris=[i for i,r in enumerate(grid["rows"]) if contains(r["box"],x,y)]
            cis=[i for i,c in enumerate(grid["columns"]) if contains(c["box"],x,y)]
            if len(ris)==1 and len(cis)==1:
                strings[ris[0]][cis[0]].append((word["order"],word["text"]))
        table=[[" ".join(t for _o,t in sorted(cell)).strip() for cell in row] for row in strings]
        headers=[norm_text(x) for x in table[0]] if table else []
        valid=bool(table and all(headers) and len(set(headers))==len(headers) and len(table)<=64 and len(table[0])<=32)
        if not valid:
            byod_outputs.append({"page_id":page["page_id"],"status":"header_or_limit_invalid","table":table,"answers":[]})
            continue

        answers=[]
        for q in q_by_page.get(page["page_id"],[]):
            result=tapas_answer(table,q["question"])
            row={"id":q["id"],"question":q["question"],"answer":result["answer"],
                 "aggregation":result["aggregation"],"cells":result["cells"]}
            accepted=(q.get("accepted_answer") or "").strip()
            if accepted:
                predicted=result["numeric_answer"] if result["aggregation"]!="NONE" else ", ".join(result["cells"])
                row["accepted_answer"]=accepted
                row["exact_string_match"]=norm_cell(predicted)==norm_cell(accepted)
            answers.append(row)
        byod_outputs.append({"page_id":page["page_id"],"status":"ok","table":table,"answers":answers})

    byod_result={"pages":len(byod_outputs),"outputs":byod_outputs}
    print({"byod_pages":len(byod_outputs),"status_counts":dict(__import__("collections").Counter(x["status"] for x in byod_outputs))})
else:
    print("BYOD disabled on canonical Run all path.")

## 29. Interpretation and limitations

### Table Intelligence is a pipeline

Detection, structure, text extraction, reconstruction and QA are separable failure points.

### Geometry is not text

Table Transformer predicts boxes. A real deployment still needs PDF text extraction, OCR or another document-extraction model.

### Gold-table QA is an upper-stage diagnostic

Strong TAPAS performance on the gold logical table does not imply strong end-to-end performance.

### Structure errors alter semantics

Missing or duplicated rows/columns can turn a correct source table into a different structured table before TAPAS sees it.

### Complex tables need richer reconstruction

Spanning cells, projected headers and nested structure require more than simple row × column intersection.

### Tutorial evidence is not a benchmark

This notebook uses a small, deterministic public-domain scientific-table sample and one seeded composition policy. Results are workshop measurements, not production-quality claims.

## 30. Terminal summary

The final cell verifies required outputs and prints the stage waterfall without declaring a winner.

In [ ]:
required=[
    out_dir/"table_detection.csv",
    out_dir/"structure_metrics.csv",
    out_dir/"reconstruction_metrics.csv",
    out_dir/"qa_questions.jsonl",
    out_dir/"qa_predictions.csv",
    out_dir/"waterfall.json",
    out_dir/"resource_metrics.csv",
    out_dir/"input_manifest.json",
    out_dir/"provenance.json",
]
missing=[str(p) for p in required if not p.is_file()]
if missing:raise RuntimeError(f"Required outputs missing: {missing}")

print("DIMER Table Intelligence Workshop")
print("-"*34)
print(f"Canonical tables: {len(canonical_tables)}")
print(f"Questions: {len(qa_records)}")
print()
print("Stage 1 — Detection")
print(f"  AP50: {det_metrics['ap50']:.3f}")
print(f"  hit@0.50: {det_metrics['hit50']:.3f}")
print(f"  mean IoU: {det_metrics['mean_best_iou']:.3f}")
print()
print("Stage 2 — Structure")
print(f"  gold-crop exact grids: {waterfall['structure_gold_crop']['grid_exact']} / {len(canonical_tables)}")
print(f"  detected-crop exact grids: {waterfall['structure_detected_crop']['grid_exact']} / {len(canonical_tables)}")
print()
print("Stage 3 — Reconstruction")
print(f"  valid gold-crop tables: {waterfall['reconstruction']['valid_gold_crop']} / {len(canonical_tables)}")
print(f"  valid detected tables: {waterfall['reconstruction']['valid_detected_crop']} / {len(canonical_tables)}")
print()
print("Stage 4 — TAPAS")
print(f"  gold-table denotation: {waterfall['qa']['gold_table_denotation_accuracy']:.3f}")
print(f"  structure-only denotation: {waterfall['qa']['structure_only_denotation_accuracy']:.3f}")
print(f"  end-to-end denotation: {waterfall['qa']['end_to_end_denotation_accuracy']:.3f}")
print()
print(f"Outputs: {out_dir}/")